
<font color="purple" size="6"><b>SaaS Revenue & Churn Analysis</b></font>
<br>
<br>
<font color="purple" size="5"><b>Data Cleaning & Data Quality Audit</b></font>
<br>

This notebook performs data quality checks and cleaning for the SaaS datasets.

Goals:

- Identify data quality issues
- Handle missing values
- Remove duplicates
- Validate date fields
- Standardize currencies
- Validate relational integrity

Output datasets will be stored in:  Data/Cleaned/


<font color="purple" size="4"><b>Importing necessary libraries</b></font>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setting display options to show all columns in the DataFrame

pd.set_option("display.max_columns", None)

<font color="purple" size="4"><b>Load Datasets</b></font>

In [2]:
accounts = pd.read_csv("../Data/Raw/accounts.csv")

usage = pd.read_csv("../Data/Raw/usage_events.csv")

tickets = pd.read_csv("../Data/Raw/support_tickets.csv")

invoices = pd.read_csv("../Data/Raw/invoices.csv")

renewals = pd.read_csv("../Data/Raw/renewals.csv")

nps = pd.read_csv("../Data/Raw/nps_survey.csv")

<font color="purple" size="4"><b>Dataset Overview, Size and Structure</b></font>

In [10]:
datasets = {
    "accounts": accounts,
    "usage": usage,
    "tickets": tickets,
    "invoices": invoices,
    "renewals": renewals,
    "nps": nps
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    print(df.info())
    


ACCOUNTS
(505, 8)
<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   account_id       505 non-null    int64
 1   company          505 non-null    str  
 2   tier             505 non-null    str  
 3   region           505 non-null    str  
 4   arr              505 non-null    int64
 5   contract_start   505 non-null    str  
 6   contract_end     505 non-null    str  
 7   account_manager  490 non-null    str  
dtypes: int64(2), str(6)
memory usage: 31.7 KB
None

USAGE
(92370, 6)
<class 'pandas.DataFrame'>
RangeIndex: 92370 entries, 0 to 92369
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   event_id     92370 non-null  int64  
 1   account_id   92370 non-null  int64  
 2   week_start   92370 non-null  str    
 3   tier         92370 non-null  str    
 4   feature      92370 non-n

In [11]:
accounts.head()

,account_id,company,tier,region,arr,contract_start,contract_end,account_manager
0,1,"Jones, Snyder and Bass",Business,APAC,15860,2024-04-07,2025-04-07,Christopher Green
1,2,Frost Inc,Business,Europe,20191,2024-07-31,2025-07-31,Shelby Miller
2,3,"Herrera, Adkins and Andersen",Starter,North America,3638,2024-06-03,2025-06-03,Wesley Reyes
3,4,Berry-Lee,Business,APAC,28526,2025-02-06,2026-02-06,Kevin Jimenez DVM
4,5,Martin Inc,Business,Europe,29423,2024-06-06,2025-06-06,Michael Cruz


In [12]:
usage.head()

,event_id,account_id,week_start,tier,feature,usage_count
0,1,1,2024-01-01,Business,dashboard,14.0
1,2,1,2024-01-01,Business,reports,10.0
2,3,1,2024-01-01,Business,automation,7.0
3,4,1,2024-01-01,Business,integrations,3.0
4,5,1,2024-01-08,Business,dashboard,13.0


In [13]:
tickets.head()

,ticket_id,account_id,created_at,category,resolution_hours,sentiment_score
0,1,112.0,2025-05-11,performance,62,-0.910419
1,2,106.0,2025-06-20,performance,16,-0.252450
2,3,281.0,2025-04-29,feature_request,33,0.148413
3,4,NaN,2025-06-01,integration,24,-0.319757
4,5,459.0,2025-12-06,billing,9,0.401519


In [14]:
invoices.head()

,invoice_id,account_id,invoice_date,due_date,paid_date,amount,currency,discount_code,payment_terms
0,INV1,1,2024-01-01,2024-01-31,2024-02-01,1321.67,USD,PROMO10,Net 60
1,INV2,1,2024-02-01,2024-03-02,2024-03-08,1321.67,EUR,LOYALTY15,Net 15
2,INV3,1,2024-03-01,2024-03-31,2024-04-01,1321.67,EUR,NaN,Net 30
3,INV4,1,2024-04-01,2024-05-01,2024-04-27,1321.67,EUR,PROMO10,Net 15
4,INV5,1,2024-05-01,2024-05-31,2024-06-08,1321.67,USD,SUMMER20,Net 15


In [15]:
renewals.head()

,renewal_id,account_id,renewal_date,previous_arr,new_arr,outcome
0,1,1,2025-01-01,15860,15860.00,renewed
1,2,2,2025-01-01,20191,20191.00,renewed
2,3,3,2025-01-01,3638,2718.29,downgraded
3,4,4,2025-01-01,28526,0.00,churned
4,5,5,2025-01-01,29423,29423.00,renewed


In [16]:
nps.head()

,survey_id,account_id,score,survey_date
0,1,6,10,2024-12-01
1,2,8,1,2024-12-01
2,3,10,10,2024-12-01
3,4,11,1,2024-12-01
4,5,16,7,2024-12-01


<font color="purple" size="4"><b>Check Missing Values</b></font>

In [17]:
for name, df in datasets.items():

    print(f"\nMissing values in {name}")

    print(df.isna().sum())


Missing values in accounts
account_id          0
company             0
tier                0
region              0
arr                 0
contract_start      0
contract_end        0
account_manager    15
dtype: int64

Missing values in usage
event_id         0
account_id       0
week_start       0
tier             0
feature          0
usage_count    100
dtype: int64

Missing values in tickets
ticket_id             0
account_id          245
created_at            0
category              0
resolution_hours      0
sentiment_score       0
dtype: int64

Missing values in invoices
invoice_id          0
account_id          0
invoice_date        0
due_date            0
paid_date           0
amount              0
currency            0
discount_code    1503
payment_terms       0
dtype: int64

Missing values in renewals
renewal_id      0
account_id      0
renewal_date    0
previous_arr    0
new_arr         0
outcome         0
dtype: int64

Missing values in nps
survey_id      0
account_id     0
sc

<font color="purple" size="4"><b>Handling Missing Values</b></font>

Different columns require different treatments:

accounts.account_manager → fill with "Unknown"

usage.usage_count → replace with 0 (no usage recorded)

tickets.account_id → remove invalid records

invoices.discount_code → fill with "No_Discount"

In [ ]:
# Fill missing values in 'account_manager' with 'Unknown'

accounts["account_manager"] = accounts["account_manager"].fillna("Unknown")
accounts.isna().sum()

account_id         0
company            0
tier               0
region             0
arr                0
contract_start     0
contract_end       0
account_manager    0
dtype: int64

In [ ]:
# Fill missing values in 'usage_count' with 0

usage["usage_count"] = usage["usage_count"].fillna(0)
usage.isna().sum()

event_id       0
account_id     0
week_start     0
tier           0
feature        0
usage_count    0
dtype: int64

In [ ]:
# Drop rows with missing 'account_id' in tickets

tickets = tickets.dropna(subset=["account_id"])
tickets.isna().sum()

ticket_id           0
account_id          0
created_at          0
category            0
resolution_hours    0
sentiment_score     0
dtype: int64

In [ ]:
# Fill missing values in 'discount_code' with 'No_Discount'

invoices["discount_code"] = invoices["discount_code"].fillna("No_Discount")
invoices.isna().sum()

invoice_id       0
account_id       0
invoice_date     0
due_date         0
paid_date        0
amount           0
currency         0
discount_code    0
payment_terms    0
dtype: int64

<font color="purple" size="4"><b>Verify Missing Values After Cleaning</b></font>

In [23]:
datasets_clean = {
    "accounts": accounts,
    "usage": usage,
    "tickets": tickets,
    "invoices": invoices,
    "renewals": renewals,
    "nps": nps
}

for name, df in datasets_clean.items():

    print(f"\n{name}")

    print(df.isna().sum())


accounts
account_id         0
company            0
tier               0
region             0
arr                0
contract_start     0
contract_end       0
account_manager    0
dtype: int64

usage
event_id       0
account_id     0
week_start     0
tier           0
feature        0
usage_count    0
dtype: int64

tickets
ticket_id           0
account_id          0
created_at          0
category            0
resolution_hours    0
sentiment_score     0
dtype: int64

invoices
invoice_id       0
account_id       0
invoice_date     0
due_date         0
paid_date        0
amount           0
currency         0
discount_code    0
payment_terms    0
dtype: int64

renewals
renewal_id      0
account_id      0
renewal_date    0
previous_arr    0
new_arr         0
outcome         0
dtype: int64

nps
survey_id      0
account_id     0
score          0
survey_date    0
dtype: int64


<font color="purple" size="4"><b>Duplicate Detection</b></font>

In [25]:
for name, df in datasets_clean.items():

    print(f"{name} duplicates:", df.duplicated().sum())

accounts duplicates: 5
usage duplicates: 50
tickets duplicates: 25
invoices duplicates: 20
renewals duplicates: 0
nps duplicates: 0


<font color="purple" size="4"><b>Remove Duplicates</b></font>

In [26]:
accounts = accounts.drop_duplicates()

usage = usage.drop_duplicates()

tickets = tickets.drop_duplicates()

invoices = invoices.drop_duplicates()

renewals = renewals.drop_duplicates()

nps = nps.drop_duplicates()

<font color="purple" size="4"><b>Convert Date Columns</b></font>

In [27]:
accounts["contract_start"] = pd.to_datetime(accounts["contract_start"])

accounts["contract_end"] = pd.to_datetime(accounts["contract_end"])

usage["week_start"] = pd.to_datetime(usage["week_start"])

tickets["created_at"] = pd.to_datetime(tickets["created_at"])

invoices["invoice_date"] = pd.to_datetime(invoices["invoice_date"])

invoices["due_date"] = pd.to_datetime(invoices["due_date"])

invoices["paid_date"] = pd.to_datetime(invoices["paid_date"])

renewals["renewal_date"] = pd.to_datetime(renewals["renewal_date"])

nps["survey_date"] = pd.to_datetime(nps["survey_date"])

<font color="purple" size="4"><b>Detect Impossible Dates</b></font>

In [28]:
bad_payments = invoices[invoices["paid_date"] < invoices["invoice_date"]]

bad_payments.head()

,invoice_id,account_id,invoice_date,due_date,paid_date,amount,currency,discount_code,payment_terms
167,INV168,14,2024-12-01,2024-12-31,2024-11-28,1408.25,EUR,No_Discount,Net 30
343,INV344,29,2024-08-01,2024-08-31,2024-07-29,972.92,EUR,No_Discount,Net 15
544,INV545,46,2024-05-01,2024-05-31,2024-04-28,348.50,EUR,PROMO10,Net 60
596,INV597,50,2024-09-01,2024-10-01,2024-08-29,1477.58,USD,PROMO10,Net 30
961,INV962,81,2024-02-01,2024-03-02,2024-01-29,166.75,USD,LOYALTY15,Net 15


<font color="purple" size="4"><b>Fix Impossible Dates</b></font>

In [ ]:
# For rows where 'paid_date' is before 'invoice_date', set 'paid_date' to 'due_date'

invoices.loc[
    invoices["paid_date"] < invoices["invoice_date"],
    "paid_date"
] = invoices["due_date"]

<font color="purple" size="4"><b>Currency Standardization</b></font>

In [ ]:
# Convert USD to EUR using a fixed exchange rate - A static FX rate is used for simplicity and reproducibility.
usd_to_eur = 0.92

invoices["amount_eur"] = np.where(
    invoices["currency"] == "USD",
    invoices["amount"] * usd_to_eur,
    invoices["amount"]
)

<font color="purple" size="4"><b>Referential Integrity Check</b></font>

In [31]:
valid_accounts = set(accounts["account_id"])

invalid_renewals = renewals[
    ~renewals["account_id"].isin(valid_accounts)
]

invalid_renewals

,renewal_id,account_id,renewal_date,previous_arr,new_arr,outcome
505,10000,10016,2025-01-01,11573,5706.0,churned
506,10001,10095,2025-01-01,17106,6262.0,renewed
507,10002,10061,2025-01-01,19357,17012.0,renewed
508,10003,10029,2025-01-01,17507,5465.0,churned
509,10004,10052,2025-01-01,10212,17187.0,churned
510,10005,10041,2025-01-01,19118,5036.0,churned
511,10006,10086,2025-01-01,14715,13700.0,churned
512,10007,10015,2025-01-01,6982,7266.0,renewed
513,10008,10030,2025-01-01,16706,4173.0,churned
514,10009,10091,2025-01-01,9355,13116.0,churned


<font color="purple" size="4"><b>Orphan Records in Renewals</b></font>

Some renewal records reference account_ids that do not exist in the accounts table.
These are considered data integrity issues.

In [34]:
len(invalid_renewals)

10

<font color="purple" size="4"><b>Remove Orphan Records</b></font>

Renewal records without a valid account_id cannot be used for analysis. These records will be removed.

In [35]:
renewals = renewals[
    renewals["account_id"].isin(valid_accounts)
]

In [ ]:
# check that all remaining renewals have valid account_ids

renewals[~renewals["account_id"].isin(valid_accounts)]

,renewal_id,account_id,renewal_date,previous_arr,new_arr,outcome


<font color="purple" size="4"><b>Final Dataset Shapes</b></font>

In [37]:
for name, df in datasets_clean.items():
    print(name, df.shape)

accounts (505, 8)
usage (92370, 6)
tickets (1785, 6)
invoices (6080, 9)
renewals (515, 6)
nps (157, 4)


In [38]:
# Final dataset shapes after cleaning

datasets_clean = {
    "accounts": accounts,
    "usage": usage,
    "tickets": tickets,
    "invoices": invoices,
    "renewals": renewals,
    "nps": nps
}

<font color="purple" size="4"><b>Save Clean Datasets</b></font>

In [ ]:
accounts.to_csv("../Data/Cleaned/accounts_clean.csv", index=False)

usage.to_csv("../Data/Cleaned/usage_clean.csv", index=False)

tickets.to_csv("../Data/Cleaned/tickets_clean.csv", index=False)

invoices.to_csv("../Data/Cleaned/invoices_clean.csv", index=False)

renewals.to_csv("../Data/Cleaned/renewals_clean.csv", index=False)

nps.to_csv("../Data/Cleaned/nps_clean.csv", index=False)

In [ ]:
# Final check for duplicates after cleaning

for name, df in datasets_clean.items():

    print(f"{name} duplicates:", df.duplicated().sum())

accounts duplicates: 0
usage duplicates: 0
tickets duplicates: 0
invoices duplicates: 0
renewals duplicates: 0
nps duplicates: 0
